# LLM12: LLM Inference — Decoding Strategies & KV-Cache

## Lab Overview

This lab explores how Large Language Models generate text at inference time. We cover the two-stage inference pipeline (**Prefill** and **Decode**), decoding strategies (greedy, beam search, sampling with temperature/top-k/top-p), and the critical **KV-Cache** optimization that makes autoregressive generation practical.

> **Quick terms:** **Prefill** runs the model once on the entire prompt (many tokens in parallel). **Decode** generates **one new token at a time** autoregressively. **KV-Cache** stores past **key/value** tensors from attention so earlier tokens are not recomputed each step.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

Implement decoding strategies from scratch, understand the FLOPs analysis of prefill vs. decode, and see how KV-Cache reduces redundant computation.

1. **Describe the Two-Stage Pipeline**: Prefill (process entire prompt) and Decode (generate token by token).
2. **Implement Decoding Strategies**: Greedy, Top-K sampling, Top-P (nucleus) sampling, and temperature scaling.
3. **Understand Repetition Penalties**: How `repetition_penalty`, `presence_penalty`, and `frequency_penalty` work.
4. **Analyze Compute Cost**: FLOPs for self-attention and MLP in terms of batch, sequence length, and hidden dim.
5. **Implement KV-Cache**: Cache Key/Value tensors across decoding steps to avoid recomputation.

---


## 1. Environment Setup


In [1]:
import math
import time
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
PyTorch version: 2.9.1+rocm7.2.0.git7e1940d4
GPU: AMD Radeon(TM) 8060S Graphics
GPU Memory: 102.7 GB


Load librocdxg.so successully!
Load all DTIF APIs OK!


## 2. LLM Inference Pipeline Overview

An LLM takes a token sequence as input, passes it through $N$ Decoder layers, and outputs **logits** (unnormalized scores over the vocabulary). A **decoding strategy** converts logits into the next token.

### Two-Stage Inference:

| Stage       | Input                   | Output                             | Characteristic                                  |
| ----------- | ----------------------- | ---------------------------------- | ----------------------------------------------- |
| **Prefill** | Full prompt (s tokens)  | Logits for next token + cached K,V | Compute-bound: processes all tokens in parallel |
| **Decode**  | One new token at a time | Next token logits                  | Memory-bound: sequential, uses KV-Cache         |

### FLOPs Analysis (per Decoder Layer):

- **Self-Attention**: $8bsh^2 + 4bs^2h$ (Q/K/V projections + attention + output projection)
- **MLP (4× expansion)**: $16bsh^2$
- **Total per layer**: $24bsh^2 + 4bs^2h$

Where $b$ = batch size, $s$ = sequence length, $h$ = hidden dimension.

> **Note:** These FLOPs formulas are simplified back-of-the-envelope estimates for teaching.
> Exact counts depend on implementation details such as bias terms, fused kernels, grouped-query attention (GQA), and the exact MLP expansion ratio.


In [2]:
def compute_flops_per_layer(b, s, h):
    """Compute FLOPs for one Decoder layer."""
    attn_flops = 8 * b * s * h**2 + 4 * b * s**2 * h
    mlp_flops = 16 * b * s * h**2
    total = attn_flops + mlp_flops
    return attn_flops, mlp_flops, total


# LLaMA-3.2-1B-like config
b, h = 1, 2048
print("=== FLOPs per Decoder Layer (LLaMA-1B-like) ===")
print(f"{'seq_len':>8s} | {'Attention':>14s} | {'MLP':>14s} | {'Total':>14s} | {'Attn %':>7s}")
print("-" * 68)
for s in [128, 512, 1024, 2048, 4096, 8192]:
    attn, mlp, total = compute_flops_per_layer(b, s, h)
    print(f"{s:>8d} | {attn:>14,.0f} | {mlp:>14,.0f} | {total:>14,.0f} | {attn / total * 100:>6.1f}%")

print("\nNote: As sequence length grows, the attention score/mixing term grows as O(s^2),")
print("so attention becomes increasingly important. In practice, whether it fully dominates")
print("depends on hidden size, head configuration, and implementation details.")

=== FLOPs per Decoder Layer (LLaMA-1B-like) ===
 seq_len |      Attention |            MLP |          Total |  Attn %
--------------------------------------------------------------------
     128 |  4,429,185,024 |  8,589,934,592 | 13,019,119,616 |   34.0%
     512 | 19,327,352,832 | 34,359,738,368 | 53,687,091,200 |   36.0%
    1024 | 42,949,672,960 | 68,719,476,736 | 111,669,149,696 |   38.5%
    2048 | 103,079,215,104 | 137,438,953,472 | 240,518,168,576 |   42.9%
    4096 | 274,877,906,944 | 274,877,906,944 | 549,755,813,888 |   50.0%
    8192 | 824,633,720,832 | 549,755,813,888 | 1,374,389,534,720 |   60.0%

Note: As sequence length grows, the attention score/mixing term grows as O(s^2),
so attention becomes increasingly important. In practice, whether it fully dominates
depends on hidden size, head configuration, and implementation details.


## 3. Decoding Strategies

Given logits $z \in \mathbb{R}^{|V|}$ from the LM head, we need to pick the next token.

### Temperature Scaling

$$\tilde{p}_i = \text{softmax}(z_i / T)$$

- $T < 1$: sharper distribution (more deterministic)
- $T > 1$: flatter distribution (more random)
- As $T \to 0^+$, sampling approaches greedy decoding
- As $T \to \infty$, the softmax distribution approaches uniform over the candidate tokens

### Top-K Sampling

Keep only the $k$ highest-probability tokens, zero out the rest, re-normalize.

### Top-P (Nucleus) Sampling

Keep the smallest set of tokens whose cumulative probability $\geq p$, zero out the rest.

**Typical pipeline**: Temperature → Top-K → Top-P → Sample


In [3]:
def greedy_decode(logits: torch.Tensor) -> int:
    """Select the token with the highest logit."""
    return logits.argmax(dim=-1).item()


def temperature_scale(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    """Scale logits by temperature."""
    if temperature <= 0:
        raise ValueError("Temperature must be > 0")
    return logits / temperature


def top_k_filter(logits: torch.Tensor, k: int) -> torch.Tensor:
    """Set all non-top-k logits to -inf so they receive zero probability after softmax."""
    if k <= 0 or k >= logits.size(-1):
        return logits
    topk_vals, _ = torch.topk(logits, k)
    threshold = topk_vals[..., -1].unsqueeze(-1)
    return logits.masked_fill(logits < threshold, float("-inf"))


def top_p_filter(logits: torch.Tensor, p: float) -> torch.Tensor:
    """Keep the smallest set of tokens whose cumulative probability reaches p."""
    if not (0.0 < p <= 1.0):
        raise ValueError("top_p must be in (0, 1].")

    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cumulative_probs = sorted_probs.cumsum(dim=-1)

    # Remove tokens whose inclusion would push cumulative probability past p,
    # while always keeping at least the first token.
    mask = cumulative_probs - sorted_probs >= p
    sorted_logits = sorted_logits.masked_fill(mask, float("-inf"))

    output = logits.clone()
    output.scatter_(-1, sorted_idx, sorted_logits)
    return output


def sample_token(logits: torch.Tensor, temperature=1.0, top_k=0, top_p=1.0) -> int:
    """Full sampling pipeline: temperature -> top_k -> top_p -> sample."""
    logits = temperature_scale(logits, temperature)
    logits = top_k_filter(logits, top_k)
    logits = top_p_filter(logits, top_p)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()


# Demonstrate with synthetic logits
vocab_size = 10
logits = torch.tensor([2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5, -2.0, -3.0])
labels = [f"tok_{i}" for i in range(vocab_size)]

print("=== Decoding Strategy Comparison ===")
print(f"Logits: {logits.tolist()}")
print(f"Softmax probs: {torch.softmax(logits, -1).tolist()}")
print(f"\nGreedy: token {greedy_decode(logits)} (always highest)")

# Sample 20 times with different strategies
for label, kwargs in [
    ("T=0.3 (sharp)", {"temperature": 0.3}),
    ("T=1.0 (default)", {"temperature": 1.0}),
    ("T=2.0 (flat)", {"temperature": 2.0}),
    ("Top-K=3", {"top_k": 3}),
    ("Top-P=0.8", {"top_p": 0.8}),
    ("T=0.7+K=5+P=0.9", {"temperature": 0.7, "top_k": 5, "top_p": 0.9}),
]:
    samples = [sample_token(logits.clone(), **kwargs) for _ in range(20)]
    unique = sorted(set(samples))
    print(f"  {label:<20s}: unique={unique}, samples={samples}")

=== Decoding Strategy Comparison ===
Logits: [2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5, -2.0, -3.0]
Softmax probs: [0.3968256711959839, 0.24068693816661835, 0.1459840089082718, 0.08854378014802933, 0.05370451509952545, 0.032573435455560684, 0.01975678652524948, 0.011983097530901432, 0.007268115878105164, 0.0026737903244793415]

Greedy: token 0 (always highest)
  T=0.3 (sharp)       : unique=[0, 1, 2], samples=[0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  T=1.0 (default)     : unique=[0, 1, 2, 3, 4, 6, 7], samples=[2, 1, 3, 6, 0, 0, 0, 1, 1, 7, 4, 0, 2, 0, 4, 2, 4, 0, 0, 1]
  T=2.0 (flat)        : unique=[0, 1, 2, 3, 4, 5, 6, 7, 9], samples=[5, 4, 7, 6, 9, 3, 6, 3, 0, 1, 5, 7, 1, 2, 2, 0, 4, 3, 1, 0]
  Top-K=3             : unique=[0, 1, 2], samples=[1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 2, 0, 1, 2, 0, 0]
  Top-P=0.8           : unique=[0, 1, 2, 3], samples=[0, 1, 2, 3, 3, 0, 0, 3, 1, 0, 3, 2, 2, 2, 0, 2, 1, 0, 0, 0]
  T=0.7+K=5+P=0.9     : unique=[0, 1, 2], sample

## 4. Repetition Penalty

Even with temperature and top-k/p, models can produce repetitive text. Penalty mechanisms modify logits based on previously generated tokens:

- **Repetition penalty** (HuggingFace-style): For each previously seen token type, adjust its logit once:
  divide by `penalty` if the logit is positive, or multiply by `penalty` if the logit is negative
- **Presence penalty** (OpenAI): Subtract a constant from any previously seen token's logit
- **Frequency penalty** (OpenAI): Subtract a value proportional to how many times the token appeared


In [4]:
def apply_repetition_penalty(logits: torch.Tensor, generated_ids: list, penalty: float = 1.2) -> torch.Tensor:
    """HuggingFace-style repetition penalty."""
    if penalty <= 0:
        raise ValueError("penalty must be > 0")

    logits = logits.clone()
    for token_id in set(generated_ids):
        if logits[token_id] > 0:
            logits[token_id] /= penalty
        else:
            logits[token_id] *= penalty
    return logits


def apply_frequency_presence_penalty(
    logits: torch.Tensor, generated_ids: list, frequency_penalty: float = 0.5, presence_penalty: float = 0.3
) -> torch.Tensor:
    """OpenAI-style frequency + presence penalty."""
    logits = logits.clone()
    from collections import Counter

    counts = Counter(generated_ids)
    for token_id, count in counts.items():
        logits[token_id] -= frequency_penalty * count
        logits[token_id] -= presence_penalty  # applied once per unique token
    return logits


# Demonstrate
logits = torch.tensor([3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5])
generated = [0, 0, 1, 0]  # token 0 appears 3 times

print("=== Repetition Penalty Demo ===")
print(f"Original logits:  {logits.tolist()}")
print(f"Generated so far: {generated}")

penalized_rep = apply_repetition_penalty(logits, generated, penalty=1.5)
print(f"After rep_penalty=1.5: {penalized_rep.tolist()}")

penalized_fp = apply_frequency_presence_penalty(logits, generated, frequency_penalty=0.5, presence_penalty=0.3)
print(f"After freq=0.5, pres=0.3: {penalized_fp.tolist()}")
print(f"\nToken 0 logit: {logits[0]:.1f} -> rep: {penalized_rep[0]:.2f}, fp: {penalized_fp[0]:.2f}")
print(
    f"Token 2 logit: {logits[2]:.1f} -> rep: {penalized_rep[2]:.2f} (unchanged), fp: {penalized_fp[2]:.2f} (unchanged)"
)

=== Repetition Penalty Demo ===
Original logits:  [3.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5]
Generated so far: [0, 0, 1, 0]
After rep_penalty=1.5: [2.0, 1.6666666269302368, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5]
After freq=0.5, pres=0.3: [1.2000000476837158, 1.7000000476837158, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5]

Token 0 logit: 3.0 -> rep: 2.00, fp: 1.20
Token 2 logit: 2.0 -> rep: 2.00 (unchanged), fp: 2.00 (unchanged)


## 5. Prefill & Decode — Step by Step

Let's build a minimal Transformer decoder and trace the two-stage pipeline explicitly.

> **Teaching simplification:** The tiny model below omits positional embeddings / RoPE.
> That is fine for demonstrating prefill, decode, and KV-Cache mechanics, but real LLMs need positional information to represent token order.


In [5]:
class TinyDecoderLayer(nn.Module):
    """Simplified single decoder layer with proper KV-Cache.

    Uses separate Q/K/V projections so we can cache the PROJECTED K and V
    tensors. This is the correct pattern — during decode, only the new token
    goes through K/V projection; past tokens' K/V come from cache.
    """

    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        # Separate Q/K/V projections (not nn.MultiheadAttention)
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.SiLU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x, kv_cache=None):
        """
        Args:
            x: (B, T, D) — full prompt (prefill) or single new token (decode)
            kv_cache: (cached_K, cached_V) each (B, H, S_past, Dh), or None
        Returns:
            output (B, T, D), new_kv_cache
        """
        B, T, D = x.shape
        residual = x
        x_norm = self.norm1(x)

        # Project Q/K/V for CURRENT tokens only
        Q = self.q_proj(x_norm).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K_new = self.k_proj(x_norm).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V_new = self.v_proj(x_norm).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Append to cache (the key optimization: past K/V are NOT re-projected!)
        if kv_cache is not None:
            K = torch.cat([kv_cache[0], K_new], dim=2)
            V = torch.cat([kv_cache[1], V_new], dim=2)
        else:
            K, V = K_new, V_new

        new_cache = (K.detach(), V.detach())

        # Apply a causal mask during prefill, where multiple query positions are processed together.
        # During decode we feed exactly one new token (T=1), so there are no "future" positions
        # within the current query block, and the single query is allowed to attend to all cached past tokens.
        attn_out = F.scaled_dot_product_attention(Q, K, V, is_causal=(kv_cache is None and T > 1))

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D)
        x = residual + self.out_proj(attn_out)
        x = x + self.mlp(self.norm2(x))
        return x, new_cache


class TinyLM(nn.Module):
    """Minimal language model with proper KV-Cache support."""

    def __init__(self, vocab_size, hidden_dim, num_heads, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.layers = nn.ModuleList([TinyDecoderLayer(hidden_dim, num_heads) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size, bias=False)

    def forward(self, input_ids, kv_caches=None):
        x = self.embedding(input_ids)
        new_caches = []
        for i, layer in enumerate(self.layers):
            cache = kv_caches[i] if kv_caches else None
            x, new_cache = layer(x, cache)
            new_caches.append(new_cache)
        x = self.norm(x)
        return self.lm_head(x), new_caches


# Use a LARGER model so KV-Cache savings are measurable over Python overhead
vocab_size = 1000
hidden_dim, num_heads, num_layers = 512, 8, 6
model = TinyLM(vocab_size, hidden_dim, num_heads, num_layers).to(device)
model.eval()
print(f"TinyLM: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Config: dim={hidden_dim}, heads={num_heads}, layers={num_layers}, vocab={vocab_size}")

TinyLM: 19,927,040 parameters
Config: dim=512, heads=8, layers=6, vocab=1000


In [9]:
# Two-stage generation demonstration


@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens=10, temperature=1.0, top_k=0):
    """
    Generate tokens using prefill + decode with KV-Cache.
    """
    input_ids = prompt_ids.unsqueeze(0).to(device)  # (1, seq_len)
    generated = prompt_ids.tolist()

    # === Stage 1: PREFILL ===
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    logits, kv_caches = model(input_ids, kv_caches=None)
    if device.type == "cuda":
        torch.cuda.synchronize()
    prefill_time = time.perf_counter() - t0

    # Get next token from last position
    next_logits = logits[0, -1, :]  # (vocab_size,)
    next_token = sample_token(next_logits, temperature=temperature, top_k=top_k)
    generated.append(next_token)

    print(f"Prefill: processed {input_ids.size(1)} tokens in {prefill_time * 1000:.2f} ms")

    # === Stage 2: DECODE ===
    decode_times = []
    for step in range(max_new_tokens - 1):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        new_input = torch.tensor([[next_token]], device=device)
        logits, kv_caches = model(new_input, kv_caches=kv_caches)

        if device.type == "cuda":
            torch.cuda.synchronize()
        decode_time = time.perf_counter() - t0
        decode_times.append(decode_time)

        next_logits = logits[0, -1, :]
        next_token = sample_token(next_logits, temperature=temperature, top_k=top_k)
        generated.append(next_token)

    avg_decode = np.mean(decode_times) * 1000 if decode_times else 0.0
    print(f"Decode:  generated {max_new_tokens} tokens, avg {avg_decode:.2f} ms/token")
    return generated


# Run generation
prompt = torch.tensor(list(range(1, 21)))  # 20-token prompt
print("=== Generation with KV-Cache ===")
output = generate(model, prompt, max_new_tokens=10, temperature=0.8, top_k=10)
print(f"Generated sequence length: {len(output)} (prompt={20} + generated={10})")

=== Generation with KV-Cache ===
Prefill: processed 20 tokens in 5.57 ms
Decode:  generated 10 tokens, avg 2.90 ms/token
Generated sequence length: 30 (prompt=20 + generated=10)


## 6. KV-Cache Memory Analysis

For each decoder layer, the KV-Cache stores:

$$\text{KV memory} = b \times L \times H \times s \times d_{\text{head}} \times 2 \times \text{dtype\_size}$$

where:

- $b$ = batch size
- $L$ = number of decoder layers
- $H$ = number of attention heads
- $s$ = cached sequence length
- $d_{\text{head}}$ = head dimension
- $\times 2$ accounts for both K and V


In [7]:
def kv_cache_memory(batch, layers, heads, seq_len, head_dim, dtype_bytes=2):
    """Compute KV-Cache memory in bytes. dtype_bytes=2 corresponds to FP16/BF16."""
    return batch * layers * heads * seq_len * head_dim * 2 * dtype_bytes


# LLaMA-3.2-1B config
cfg = {"batch": 1, "layers": 16, "heads": 32, "head_dim": 64}

print("=== KV-Cache Memory (LLaMA-1B-like, FP16) ===")
print(f"{'Seq Length':>10s} | {'KV Memory':>12s} | {'Per Layer':>12s}")
print("-" * 40)
for s in [512, 1024, 2048, 4096, 8192, 16384, 32768, 131072]:
    total = kv_cache_memory(seq_len=s, **cfg)
    per_layer = total / cfg["layers"]
    print(f"{s:>10,d} | {total / 1e6:>10.1f} MB | {per_layer / 1e6:>10.2f} MB")

print(f"\nWith 128K context: {kv_cache_memory(seq_len=131072, **cfg) / 1e9:.2f} GB of KV-Cache alone.")
print("This motivates sparse attention and KV compression techniques.")

=== KV-Cache Memory (LLaMA-1B-like, FP16) ===
Seq Length |    KV Memory |    Per Layer
----------------------------------------
       512 |       67.1 MB |       4.19 MB
     1,024 |      134.2 MB |       8.39 MB
     2,048 |      268.4 MB |      16.78 MB
     4,096 |      536.9 MB |      33.55 MB
     8,192 |     1073.7 MB |      67.11 MB
    16,384 |     2147.5 MB |     134.22 MB
    32,768 |     4295.0 MB |     268.44 MB
   131,072 |    17179.9 MB |    1073.74 MB

With 128K context: 17.18 GB of KV-Cache alone.
This motivates sparse attention and KV compression techniques.


> **Note:** The `generate()` helper prints prefill/decode timing information for illustration, so the benchmark output below will include those intermediate messages.


In [8]:
# Compare generation speed: with KV-Cache vs without (recompute full sequence each step)


@torch.no_grad()
def generate_no_cache(model, prompt_ids, max_new_tokens=10, temperature=1.0, top_k=0):
    """Generate WITHOUT KV-Cache — recomputes full sequence each step."""
    generated = prompt_ids.tolist()
    for _ in range(max_new_tokens):
        input_ids = torch.tensor([generated], device=device)
        logits, _ = model(input_ids, kv_caches=None)  # no cache — full recompute!
        next_logits = logits[0, -1, :]
        next_token = sample_token(next_logits, temperature=temperature, top_k=top_k)
        generated.append(next_token)
    return generated


# Benchmark with enough tokens to see the difference
prompt = torch.tensor(list(range(1, 101)))  # 100-token prompt
n_gen = 100

# Warm up GPU (first CUDA call has overhead)
_ = model(torch.tensor([[1]], device=device))
if device.type == "cuda":
    torch.cuda.synchronize()

# With KV-Cache
torch.manual_seed(42)
if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
out1 = generate(model, prompt, max_new_tokens=n_gen, temperature=0.8, top_k=10)
if device.type == "cuda":
    torch.cuda.synchronize()
time_cache = time.perf_counter() - t0

# Without KV-Cache
torch.manual_seed(42)
if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
out2 = generate_no_cache(model, prompt, max_new_tokens=n_gen, temperature=0.8, top_k=10)
if device.type == "cuda":
    torch.cuda.synchronize()
time_nocache = time.perf_counter() - t0

print("\n=== KV-Cache Speed Comparison ===")
print(f"Config: {n_gen} new tokens, {len(prompt)}-token prompt, dim={hidden_dim}, layers={num_layers}")
print(f"With KV-Cache:    {time_cache * 1000:.1f} ms total")
print(f"Without KV-Cache: {time_nocache * 1000:.1f} ms total")
if time_cache > 0:
    print(f"Speedup: {time_nocache / time_cache:.2f}×")
print("\nWhy? Without cache, each decode step re-runs K/V projections on ALL past tokens.")
print("With cache, only 1 new token goes through K/V projection per step.")

Prefill: processed 100 tokens in 9.63 ms
Decode:  generated 100 tokens, avg 0.00 ms/token

=== KV-Cache Speed Comparison ===
Config: 100 new tokens, 100-token prompt, dim=512, layers=6
With KV-Cache:    660.4 ms total
Without KV-Cache: 536.8 ms total
Speedup: 0.81×

Why? Without cache, each decode step re-runs K/V projections on ALL past tokens.
With cache, only 1 new token goes through K/V projection per step.


## Conclusions

### Technical Concepts Learned

- **Two-Stage Inference**: Prefill (parallel, compute-bound) and Decode (sequential, memory-bound)
- **Decoding Strategies**: Greedy, temperature scaling, top-k filtering, top-p (nucleus) sampling
- **Penalty Mechanisms**: Repetition, presence, and frequency penalties to avoid degenerate output
- **FLOPs Analysis**: Over a sequence of length $s$, attention includes an $O(s^2)$ score/mixing term, while MLP scales linearly with $s$ for fixed hidden size. As context grows, attention becomes increasingly expensive.
- **KV-Cache**: Cache K/V tensors from previous tokens to avoid $O(s^2)$ recomputation at each decode step
- **KV-Cache Memory**: Grows linearly with sequence length; for 128K context can exceed several GB

### Experiment Further

- Implement beam search with length penalty
- Profile a real model (TinyLlama) with and without `use_cache=True`
- Add `no_repeat_ngram_size` constraint to prevent n-gram repetition
- Measure how KV-Cache memory scales with batch size in a serving scenario
- Compare streaming generation latency (time-to-first-token vs. throughput)


---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
